<a href="https://colab.research.google.com/github/hsnheg1958-hub/Data-Analysis/blob/main/Introduction_to_Decision_Trees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Decision Trees

A **Decision Tree** is a versatile and highly interpretable machine learning algorithm used for both **Classification** (predicting labels) and **Regression** (predicting continuous values). It mimics human decision-making by splitting data into branches based on feature values.

---

### 1. Main Concepts
- **Root Node**: The top-most node of the tree, representing the entire dataset and the first decision split.
- **Decision (Internal) Nodes**: Nodes where the data splits into two or more branches based on a feature condition.
- **Leaf Nodes**: The terminal nodes that do not split further. They represent the final predicted class or value.
- **Splitting**: The process of dividing a node into sub-nodes based on specific criteria.
- **Pruning**: Removing parts of the tree that do not provide significant predictive power to avoid overfitting.

---

### 2. Goal and Usage
- **Goal**: To create a model that predicts the value of a target variable by learning simple decision rules inferred from the data features.
- **Usage**:
  - Credit risk assessment (default vs. non-default).
  - Medical diagnosis (disease vs. healthy based on symptoms).
  - Customer churn prediction in business.

### 3 & 4. Practical Steps & Example

We will build a Decision Tree classifier using the famous **Iris Flower Dataset** to classify flowers into three species based on their measurements.

In [ ]:
# Step 1: Import necessary libraries
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Step 2: Load and explore the dataset
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"Features: {feature_names}")
print(f"Target classes: {target_names}")
print(f"Dataset shape: {X.shape}")

In [ ]:
# Step 3: Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

In [ ]:
# Step 4: Initialize and train the Decision Tree Classifier
# We use 'entropy' or 'gini' to measure split quality. Let's use 'entropy'.
clf = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
clf.fit(X_train, y_train)
print("Model training complete.")

In [ ]:
# Step 5: Evaluate the model's performance
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy on Test Set: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
# Step 6: Visualize the learned Decision Tree
plt.figure(figsize=(12, 8))
plot_tree(clf,
          feature_names=feature_names,
          class_names=target_names,
          filled=True,
          rounded=True)
plt.title("Visualizing the Trained Decision Tree")
plt.show()

## Understanding Pruning to Prevent Overfitting

A major weakness of Decision Trees is their tendency to **overfit** the training data. Because a tree can continue splitting until every single training sample is in its own pure leaf node, it can grow incredibly deep and complex. While this achieves $100\%$ accuracy on the training data, it fails to generalize to new, unseen test data because it has essentially "memorized" the noise and outliers.

To combat this, we use **Pruning**—the process of cutting back branches that provide little to no predictive power.

---

### 1. Types of Pruning

There are two main strategies for pruning a tree:

#### A. Pre-Pruning (Early Stopping)
Pre-pruning halts the growth of the tree before it perfectly classifies all training instances. We set stopping criteria beforehand, such as:
* **`max_depth`**: Restricts the maximum number of levels down from the root. (In our code above, we restricted this to `max_depth=3`).
* **`min_samples_split`**: Specifies the minimum number of samples required to split an internal node.
* **`min_samples_leaf`**: Specifies the minimum number of samples required to be at a leaf node.

#### B. Post-Pruning (Cost Complexity Pruning)
Post-pruning allows the tree to grow to its full depth first, and then systematically collapses non-significant nodes from the bottom up.
* In `scikit-learn`, this is implemented using **Minimal Cost-Complexity Pruning** (parameterized by `ccp_alpha`).
* It calculates a cost function that balances tree size (complexity) against impurity (error rate). Higher values of `ccp_alpha` prune more of the tree, creating simpler models.

---

### 2. Practical Demonstration of Overfitting vs. Pruned Tree

Let's see what happens if we let the tree grow completely unconstrained (`max_depth=None`), and compare it to a pruned tree.

In [ ]:
# Train an Overfitted Tree (No constraints)
overfitted_clf = DecisionTreeClassifier(criterion='entropy', max_depth=None, random_state=42)
overfitted_clf.fit(X_train, y_train)

# Train a Pruned Tree (Constraint set)
pruned_clf = DecisionTreeClassifier(criterion='entropy', max_depth=2, random_state=42)
pruned_clf.fit(X_train, y_train)

print(f"Overfitted Tree Depth: {overfitted_clf.get_depth()}")
print(f"Pruned Tree Depth: {pruned_clf.get_depth()}")

# Compare performance
print(f"Overfitted Test Accuracy: {accuracy_score(y_test, overfitted_clf.predict(X_test)) * 100:.2f}%")
print(f"Pruned Test Accuracy: {accuracy_score(y_test, pruned_clf.predict(X_test)) * 100:.2f}%")

### Key Benefits of Pruning to Prevent Overfitting

Pruning a Decision Tree is analogous to tidying up a complex process by removing redundant rules. It has several crucial benefits in machine learning workflows:

1. **Improved Generalization (Mitigates Overfitting)**
   * **The Problem:** An unconstrained tree creates highly customized rules for every unique anomaly or noise in the training dataset.
   * **The Solution:** By pruning these highly specific nodes, the model focuses on broader, more robust patterns, ensuring it performs significantly better on completely unseen test data.

2. **Enhanced Interpretability (Occam's Razor)**
   * Decision trees are valued because humans can trace their logic easily. A tree with a depth of 15 containing dozens of splits is practically unreadable.
   * Pruned trees (e.g., depth 2 or 3) provide clean, straightforward decision paths that domain experts can verify instantly.

3. **Reduced Computational Complexity (Better Performance)**
   * Simpler, shallow trees require less memory to store and fewer conditional evaluations during inference.
   * If you are deploying your tree model into production (especially on low-power edge devices or real-time low-latency systems), pruned models run significantly faster.

4. **Robustness to Noise**
   * Real-world datasets often contain mislabeled entries or corrupted measurements. Pruning prevents the tree from creating special branches to accommodate these incorrect datapoints, maintaining a smoother and more robust decision surface.

### Comparison: Gini Impurity vs. Entropy

In decision trees, **Gini Impurity** and **Entropy** are both metrics used to measure the "purity" of nodes and determine where to split. Here is how they compare:

* **Mathematical Formulations**:
  * **Gini Impurity**: $Gini(S) = 1 - \sum (p_i)^2$
  * **Entropy**: $Entropy(S) = -\sum p_i \log_2(p_i)$

* **Computation Speed**: Gini Impurity is computationally faster to calculate because it does not require calculating logarithmic functions ($
\log_2$). For massive datasets, Gini is typically preferred for this performance benefit.

* **Performance Differences**: In $99\%$ of real-world scenarios, both criteria produce nearly identical trees and achieve matching accuracy scores. Gini Impurity tends to isolate the most frequent class in its own branch, whereas Entropy attempts to produce more balanced splits.

Let's test this directly by training a secondary model with Gini Impurity and evaluating its accuracy.

In [ ]:
# Train a secondary model using Gini Impurity
clf_gini = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
clf_gini.fit(X_train, y_train)

# Evaluate the Gini model
y_pred_gini = clf_gini.predict(X_test)
accuracy_gini = accuracy_score(y_test, y_pred_gini)

print(f"Entropy Model Accuracy: {accuracy * 100:.2f}%")
print(f"Gini Model Accuracy:    {accuracy_gini * 100:.2f}%")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Train both models
model_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
model_entropy.fit(X_train, y_train)

model_gini = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
model_gini.fit(X_train, y_train)

# Evaluate accuracy
acc_entropy = accuracy_score(y_test, model_entropy.predict(X_test))
acc_gini = accuracy_score(y_test, model_gini.predict(X_test))

# Create a visual comparison
comparison_df = pd.DataFrame({
    'Criterion': ['Entropy', 'Gini Impurity'],
    'Accuracy': [acc_entropy, acc_gini]
})

plt.figure(figsize=(6, 4))
# We assign hue='Criterion' and set legend=False to resolve the FutureWarning cleanly
sns.barplot(x='Criterion', y='Accuracy', hue='Criterion', data=comparison_df, palette='Set2', legend=False)
plt.ylim(0.9, 1.01)
plt.title('Accuracy Comparison: Entropy vs Gini')
for index, row in comparison_df.iterrows():
    plt.text(index, row['Accuracy'] + 0.002, f"{row['Accuracy']*100:.2f}%", color='black', ha="center")
plt.show()

### Visualizing Decision Boundaries: Entropy vs. Gini

To visualize decision boundaries in a 2D plot, we will restrict our training features to just **Petal Length** and **Petal Width** (the most critical features for classifying Iris species). This allows us to map the boundaries clearly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# We use index 2 (petal length) and index 3 (petal width)
X_2d_train = X_train[:, [2, 3]]
X_2d_test = X_test[:, [2, 3]]

# Retrain models on 2D data
clf_entropy_2d = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
clf_entropy_2d.fit(X_2d_train, y_train)

clf_gini_2d = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
clf_gini_2d.fit(X_2d_train, y_train)

def plot_boundaries(clf, X, y, title, ax):
    # Setup marker generator and color map
    markers = ('o', 's', '^')
    colors = ('#FF9999', '#99FF99', '#9999FF')
    cmap = ListedColormap(colors[:len(np.unique(y))])

    # Plot the decision surface
    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, 0.02),
                           np.arange(x2_min, x2_max, 0.02))

    Z = clf.predict(np.array([xx1.ravel(), xx2.ravel()]).T)
    Z = Z.reshape(xx1.shape)

    ax.contourf(xx1, xx2, Z, alpha=0.3, cmap=cmap)
    ax.set_xlim(xx1.min(), xx1.max())
    ax.set_ylim(xx2.min(), xx2.max())

    # Plot class samples
    for idx, cl in enumerate(np.unique(y)):
        ax.scatter(x=X[y == cl, 0],
                   y=X[y == cl, 1],
                   alpha=0.8,
                   c=[colors[idx]],
                   marker=markers[idx],
                   label=target_names[cl],
                   edgecolor='black')

    ax.set_title(title)
    ax.set_xlabel('Petal Length (cm)')
    ax.set_ylabel('Petal Width (cm)')
    ax.legend(loc='upper left')

# Create subplots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_boundaries(clf_entropy_2d, X_2d_train, y_train, 'Decision Boundary (Entropy, max_depth=3)', axes[0])
plot_boundaries(clf_gini_2d, X_2d_train, y_train, 'Decision Boundary (Gini Impurity, max_depth=3)', axes[1])

plt.tight_layout()
plt.show()

### Increasing `max_depth` to 5

By increasing the maximum depth, we allow the decision tree to make more sequential splits, carving the space into smaller and more complex rectangular regions. Let's see how this affects our boundaries.

In [ ]:
# Retrain models on 2D data with max_depth=5
clf_entropy_5 = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)
clf_entropy_5.fit(X_2d_train, y_train)

clf_gini_5 = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)
clf_gini_5.fit(X_2d_train, y_train)

# Create subplots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_boundaries(clf_entropy_5, X_2d_train, y_train, 'Decision Boundary (Entropy, max_depth=5)', axes[0])
plot_boundaries(clf_gini_5, X_2d_train, y_train, 'Decision Boundary (Gini Impurity, max_depth=5)', axes[1])

plt.tight_layout()
plt.show()

### Tuning Decision Tree Hyperparameters

To build a robust decision tree, we can tune several key hyperparameters beyond just `max_depth` and `criterion`:

* **`max_depth`**: The maximum depth of the tree. Shallower trees avoid overfitting.
* **`min_samples_split`**: The minimum number of samples required to split an internal node. Higher values prevent the model from learning highly specific rules.
* **`min_samples_leaf`**: The minimum number of samples required to be at a leaf node. This smooths the model, especially for regression or noisy data.
* **`max_features`**: The number of features to consider when looking for the best split.

Instead of guessing these values, we can use `GridSearchCV` to automatically search through a grid of parameter combinations and select the best one using cross-validation.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 3, 4, 5, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize the Decision Tree Classifier
base_clf = DecisionTreeClassifier(random_state=42)

# Set up Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(estimator=base_clf, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Fit the grid search to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and estimator
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print("Best Hyperparameters found:")
for param, val in best_params.items():
    print(f"  {param}: {val}")

# Evaluate the optimized model on the test set
y_grid_pred = best_model.predict(X_test)
optimized_accuracy = accuracy_score(y_test, y_grid_pred)
print(f"\nOptimized Model Test Accuracy: {optimized_accuracy * 100:.2f}%")

## Understanding Hyperparameters and How to Use Them

### What are Hyperparameters?
In machine learning, we distinguish between two types of parameters:
1. **Model Parameters**: These are internal parameters learned by the model during training from the dataset (e.g., the specific feature splits and thresholds in our Decision Tree).
2. **Hyperparameters**: These are external configurations set by the user *before* the training process begins (such as `max_depth`, `criterion`, `min_samples_split`). They control the learning process and dictate model complexity.

### Why Hyperparameters Matter
* **Preventing Overfitting/Underfitting**: As demonstrated by our earlier experiment, setting the right boundaries (like constraining `max_depth` or specifying `min_samples_split`) prevents the tree from capturing noise and helps it generalize better to unseen data.
* **Optimizing Performance**: Different datasets require different model shapes. Finding the sweet spot maximizes the model's accuracy and reliability on real-world test sets.

### How to Tune Hyperparameters
1. **Validation Splits**: Never tune hyperparameters on your test set, as this causes data leakage. Always use a validation split or cross-validation on the training set.
2. **Grid Search (`GridSearchCV`)**: As shown in the cell above, Grid Search systematically evaluates every permutation of your specified hyperparameter dictionary and finds the absolute best combination using cross-validation.
3. **Randomized Search (`RandomizedSearchCV`)**: Instead of evaluating all combinations, this approach randomly samples a fixed number of configurations from a distribution. It is highly efficient for large hyperparameter spaces.

## Understanding Hyperparameters and How to Use Them

### What are Hyperparameters?
In machine learning, we distinguish between two types of parameters:
1. **Model Parameters**: These are internal parameters learned by the model during training from the data (e.g., the specific feature splits and thresholds in a Decision Tree, or weights in a neural network).
2. **Hyperparameters**: These are external configurations set by the practitioner *before* the training process begins. They govern how the training process runs and directly control the complexity and behavior of the model.

### Why are Hyperparameters Important?
* **Preventing Overfitting/Underfitting**: As seen in the decision tree example, setting constraints like `max_depth` or `min_samples_split` prevents the model from fitting too closely to the noise in the training set.
* **Optimizing Performance**: Different datasets require different configurations. Finding the sweet spot optimizes predictive performance (e.g., accuracy, precision, or F1-score).

### How to Choose and Use Hyperparameters
1. **Heuristics & Defaults**: Starting with standard defaults provided by libraries like `scikit-learn` is a good baseline.
2. **Validation Splits**: Never tune hyperparameters on your test dataset. Always use a separate validation dataset or cross-validation on the training set.
3. **Grid Search (`GridSearchCV`)**: A systematic method that evaluates all combinations from a predefined grid of hyperparameters to find the best-performing set using cross-validation.
4. **Randomized Search (`RandomizedSearchCV`)**: Randomly samples combinations from specified distributions, which is computationally faster when the hyperparameter space is very large.

---

## Appendix: Mathematical Aspects (Simplified)

Decision Trees decide how to split the data by finding the questions that reduce uncertainty or impurity the most.

### A. Entropy
Entropy measures the degree of impurity, disorder, or randomness in a group of data points.

$$\text{Entropy}(S) = - \sum_{i=1}^{c} p_i \log_2(p_i)$$

- Where $p_i$ is the probability/proportion of class $i$ in the dataset $S$.
- If a node is completely pure (contains only one class), its Entropy is **0**.
- If a node is evenly split among classes, its Entropy is at its maximum (**1** for binary classification).

### B. Information Gain
Information Gain measures how much Entropy is reduced after a dataset is split on a specific feature. The tree algorithm selects the feature split that yields the highest Information Gain.

$$\text{Information Gain}(S, A) = \text{Entropy}(S) - \sum_{v \in \text{Values}(A)} \frac{|S_v|}{|S|} \text{Entropy}(S_v)$$

Essentially:
$$\text{Information Gain} = \text{Entropy before split} - \text{Weighted Entropy after split}$$

By maximizing this gain, the tree ensures that each step brings us closer to clean, pure categories!